In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import re
import yt_dlp


def _norm_date(date_str: str | None) -> str | None:
    """Convertit 'YYYY-MM-DD' ou 'YYYYMMDD' -> 'YYYYMMDD'. Retourne None si vide/invalide."""
    if not date_str:
        return None
    s = date_str.strip().replace("-", "")
    return s if len(s) == 8 and s.isdigit() else None


def _contains_keyword(text: str | None, keyword: str) -> bool:
    """Recherche insensible à la casse, robuste à None."""
    if not keyword:
        return False
    return keyword.lower() in (text or "").lower()


def _safe_filename(s: str, max_len: int = 160) -> str:
    s = (s or "").strip()
    s = re.sub(r"[\\/:*?\"<>|]", "_", s)
    s = re.sub(r"\s+", " ", s)
    return s[:max_len].strip() if len(s) > max_len else s


class YouTubeChannelSearcher:
    def __init__(self, output_dir: str = "resultats_recherche"):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

    def obtenir_videos_chaine(self, url_chaine: str) -> list[dict]:
        """Listing 'flat' des vidéos d'une chaîne (souvent sans date/description/durée/vues)."""
        url_videos = url_chaine.rstrip("/") + "/videos"
        ydl_opts = {"quiet": True, "no_warnings": True, "extract_flat": True}

        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                info = ydl.extract_info(url_videos, download=False) or {}
            entries = info.get("entries") or []
            return [e for e in entries if isinstance(e, dict)]
        except Exception as e:
            print(f"Erreur lors du listing de la chaîne: {e}")
            return []

    def _obtenir_details_video(self, video: dict) -> dict | None:
        """Récupère les détails complets d'une vidéo (à partir d'une entrée flat id/url)."""
        vid = video.get("id")
        url = video.get("url") or (f"https://www.youtube.com/watch?v={vid}" if vid else None)
        if not url:
            return None

        ydl_opts = {"quiet": True, "no_warnings": True}
        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                details = ydl.extract_info(url, download=False)
            return details if isinstance(details, dict) else None
        except Exception:
            return None

    def rechercher_videos(
        self,
        url_chaine: str,
        mot_cle: str,
        date_debut: str | None = None,
        date_fin: str | None = None,
        dans_titre: bool = True,
        dans_description: bool = False,
        prefiltre_titre: bool = True,
        max_candidats: int | None = 400,
        strict_date: bool = False,
    ) -> list[dict]:
        """
        Pipeline robuste (liste -> candidats -> détails -> filtre final).

        prefiltre_titre:
          - True: on garde uniquement les entrées flat dont le titre contient le mot-clé (rapide).
          - False: on enrichit (détails) beaucoup plus de vidéos, utile si on cherche en description.

        strict_date:
          - False (défaut): si upload_date manque, on garde la vidéo (on ne peut pas trancher).
          - True: si upload_date manque, on exclut la vidéo quand un filtre date est demandé.
        """
        d0 = _norm_date(date_debut)
        d1 = _norm_date(date_fin)
        need_date_filter = bool(d0 or d1)

        videos_flat = self.obtenir_videos_chaine(url_chaine)
        if not videos_flat:
            return []

        # Préfiltre rapide sur titre "flat" (optionnel, séparé de dans_titre)
        candidats = videos_flat
        if prefiltre_titre and mot_cle:
            candidats = [v for v in candidats if _contains_keyword(v.get("title"), mot_cle)]

        if max_candidats is not None:
            candidats = candidats[: max_candidats]

        resultats: list[dict] = []

        for v in tqdm(candidats, desc="Récupération des détails", unit="vidéo"):
            details = self._obtenir_details_video(v)
            if not details:
                continue

            upload_date = details.get("upload_date")  # 'YYYYMMDD' quand dispo

            # Filtre date
            if need_date_filter:
                if not upload_date:
                    if strict_date:
                        continue
                else:
                    if d0 and upload_date < d0:
                        continue
                    if d1 and upload_date > d1:
                        continue

            # Filtre mot-clé final (sur détails complets)
            titre_ok = _contains_keyword(details.get("title"), mot_cle) if dans_titre else False
            desc_ok = _contains_keyword(details.get("description"), mot_cle) if dans_description else False

            if (dans_titre and titre_ok) or (dans_description and desc_ok):
                resultats.append(details)

        return resultats

    def afficher_resultats(self, videos: list[dict]) -> None:
        print(f"\nRésultats: {len(videos)} vidéo(s)\n")
        for i, v in enumerate(videos, 1):
            title = v.get("title") or "N/A"
            upload_date = v.get("upload_date") or "N/A"
            if upload_date != "N/A" and isinstance(upload_date, str) and len(upload_date) == 8:
                upload_date = f"{upload_date[:4]}-{upload_date[4:6]}-{upload_date[6:]}"
            duration = v.get("duration") or 0
            mm, ss = divmod(int(duration), 60) if duration else (0, 0)
            vid = v.get("id") or "N/A"
            url = v.get("webpage_url") or (f"https://www.youtube.com/watch?v={vid}" if vid != "N/A" else "N/A")

            print(f"{i}. {title}")
            print(f"   Date: {upload_date}")
            print(f"   Durée: {mm}:{ss:02d}")
            print(f"   URL: {url}\n")

    def sauvegarder_resultats(self, videos: list[dict], fichier_sortie: str = "videos_breaking.json") -> Path:
        """Sauvegarde une version allégée (mais fiable) des détails dans un JSON."""
        fichier = self.output_dir / fichier_sortie

        resultats = []
        for v in videos:
            desc = v.get("description") or ""
            vid = v.get("id")
            url = v.get("webpage_url") or (f"https://www.youtube.com/watch?v={vid}" if vid else None)

            resultats.append(
                {
                    "id": vid,
                    "title": v.get("title"),
                    "url": url,
                    "upload_date": v.get("upload_date"),
                    "duration": v.get("duration"),
                    "view_count": v.get("view_count"),
                    "description_200": (desc[:200] + "...") if desc else "",
                }
            )

        with open(fichier, "w", encoding="utf-8") as f:
            json.dump(resultats, f, indent=2, ensure_ascii=False)

        return fichier

    def telecharger_videos_filtrees(self, videos: list[dict], qualite: str = "best") -> None:
        """
        Télécharge toutes les vidéos filtrées.
        Par défaut "best" = MP4-friendly : bestvideo(mp4)+bestaudio(m4a) sinon best mp4.
        """
        if not videos:
            print("Aucune vidéo à télécharger")
            return

        if qualite == "best":
            fmt = "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best"
        else:
            fmt = qualite

        ydl_opts = {
            "format": fmt,
            "outtmpl": str(self.output_dir / "%(upload_date)s - %(title)s.%(ext)s"),
            "merge_output_format": "mp4",
            "restrictfilenames": True,
            "noplaylist": True,
            "quiet": False,
        }

        print(f"\nTéléchargement de {len(videos)} vidéo(s) dans: {self.output_dir.resolve()}\n")

        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            for i, v in enumerate(videos, 1):
                vid = v.get("id")
                url = v.get("webpage_url") or (f"https://www.youtube.com/watch?v={vid}" if vid else None)
                title = _safe_filename(v.get("title") or "N/A")
                if not url:
                    print(f"[{i}/{len(videos)}] Skip (pas d'URL): {title}")
                    continue
                print(f"[{i}/{len(videos)}] Téléchargement: {title}")
                try:
                    ydl.download([url])
                    print("  OK")
                except Exception as e:
                    print(f"  Erreur: {e}")



In [6]:
def main():
    # Profil A (rapide)
    url_chaine = "https://www.youtube.com/@FoxNews"
    mot_cle = "Immigration"
    date_debut = "2024-08-12"
    date_fin = "2024-09-25"

    searcher = YouTubeChannelSearcher(output_dir="/Users/Manon/Desktop/Fac/ENC/Mémoire/Computer/Scrapping/Youtube_scraper/resultats_recherche")

    videos = searcher.rechercher_videos(
        url_chaine=url_chaine,
        mot_cle=mot_cle,
        date_debut=date_debut,
        date_fin=date_fin,
        dans_titre=True,
        dans_description=False,  # profil A
        prefiltre_titre=True,    # profil A
        max_candidats=400,       # profil A
        strict_date=False,       # profil A
    )

    searcher.afficher_resultats(videos)

    out = searcher.sauvegarder_resultats(videos, "videos_breaking.json")
    print(f"JSON écrit dans: {out}")

    # Décommentez cette ligne si vous voulez télécharger ; je ne sais pas s'il faut qualité=best
    # searcher.telecharger_videos_filtrees(videos, qualite="best")


if __name__ == "__main__":
    main()

KeyboardInterrupt: 